Genetic Algorithm for generating the dimensions of a silicon steel transformer core.

This algorithm generates a list of width and thickness values for the steps of a transformer's silicon steel core. The generated values are already discretized, so they can be directly used to build a real core. This algorithm is useful for generating cores for small and medium-sized transformers; however, for high-power transformer cores, which most of the time require cooling channels between the steps, it is not yet able to handle these cases. This is a modification that could be implemented in the future, allowing the algorithm to include these cooling channels.

This code was developed by Henriqui Ferreira Segantini as coursework for the Artificial Intelligence course of the Graduate Program in Computer Science at the Federal University of Technology – Paraná (UTFPR).

In [ ]:
import random
import math
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List
import multiprocessing

CORE CONFIGURATION

In [ ]:
# Represents the core column diameter, the algorithm will respect this limit and
# will not allow the step vertices to exceed this boundary.
CORE_DIAMETER = 300

# The number of steps in each hemisphere that the generated core will have.
NUM_STEPS = 8

# The thickness of one step-lap package, all generated thicknesses will be multiples of this value.
STEP_LAP_THICKNESS = 2.7

# The interval between the core widths, all generated widths will be multiples of this value,
# for core widths, integer multiples of 5, 10 or 20 are generally used,
# but the algorithm is also able to handle decimal values.
WIDTH_INTERVAL = 10

# Stacking factor used to calculate the magnetic cross-sectional area.
STACKING_FACTOR = 0.96

ALGORITHM CONFIGURATION

In [ ]:
POPULATION_SIZE = 200

# Number of generations the algorithm will run, the higher this number, the longer the algorithm's execution time, use multiples of 50.
GENERATIONS = 2000

MUTATION_RATE = 0.5

CLASSES

In [ ]:
@dataclass
class Step:
    width: float     # In mm
    thickness: float # In mm (must always be a multiple of the step-lap package, e.g. 2.7mm)

@dataclass
class CoreChromosome:
    steps: List[Step]
    fitness: float = 0.0

GENERATE INDIVIDUAL

In [ ]:
def generate_random_individual(max_radius: float) -> CoreChromosome:
    """
    Generates an initial core with decreasing widths and thicknesses based on the step lap.
    This logic must ensure the individual is born "almost" inside the circle to help the algorithm.
    """
    steps = []
    accumulated_y = 0.0

    # Randomly distributes the total available height among the steps
    estimated_total_height = max_radius * random.uniform(0.75, 0.99)
    base_thickness = (estimated_total_height / NUM_STEPS)

    # Rounds to the nearest multiple of the step lap (e.g. 2.7)
    base_thickness = round(base_thickness / STEP_LAP_THICKNESS) * STEP_LAP_THICKNESS

    for i in range(NUM_STEPS):
        accumulated_y += base_thickness

        # Calculates the theoretical maximum width for this Y using Pythagoras (x^2 + y^2 = R^2)
        if accumulated_y < max_radius:
            max_width = 2 * math.sqrt(max_radius**2 - accumulated_y**2)
        else:
            max_width = 10.0 # Minimum escape value

        # Adds a small random variation for the GA to have something to optimize
        width = max_width * random.uniform(0.9, 1.0)

        # Discretizes the width to be a multiple of 10
        width = round(width / WIDTH_INTERVAL) * WIDTH_INTERVAL
        if width < WIDTH_INTERVAL: # Ensures a minimum width of 10mm
            width = WIDTH_INTERVAL

        steps.append(Step(width=width, thickness=base_thickness))

    chromosome = CoreChromosome(steps=steps)
    normalize_widths(chromosome) # Normalizes the widths after the initial creation
    return chromosome

FITNESS FUNCTION

In [ ]:
def evaluate_fitness(chromosome: CoreChromosome, max_radius: float) -> float:
    """
    Calculates the useful area of the core and applies severe penalties if it hits the circle's boundary.
    """
    total_area = 0.0
    accumulated_y = 0.0
    penalty = 0.0

    for step in chromosome.steps:
        # The area considers the entire sheet (Width x Thickness)
        total_area += step.width * step.thickness

        # Coordinates of the step's top right corner (x, y)
        corner_x = step.width / 2.0
        accumulated_y += step.thickness
        corner_y = accumulated_y

        # Checks the distance from the corner to the center of the circle
        distance_to_center = math.sqrt(corner_x**2 + corner_y**2)

        # If the distance is greater than the radius, the step tore through the boundary
        if distance_to_center > max_radius:
            # Progressive penalty: the further outside, the worse the score
            excess = distance_to_center - max_radius
            penalty += (excess * 5000) # Adjustable multiplier weight

    # Actual magnetic area = (Sum of the half-column area * 2) * Stacking Factor
    magnetic_area = (total_area * 2) * STACKING_FACTOR

    # The final fitness is the area minus the geometric errors
    chromosome.fitness = magnetic_area - penalty
    return chromosome.fitness

GENETIC ALGORITHM


In [ ]:
def tournament_selection(population: List[CoreChromosome], tournament_size: int = 3) -> CoreChromosome:
    """
    Picks N random individuals from the population and returns the one with the highest score (fitness).
    """
    competitors = random.sample(population, tournament_size)
    sorted_competitors = sorted(competitors, key=lambda c: c.fitness, reverse=True)
    return sorted_competitors[0]


def single_point_crossover(parent1: CoreChromosome, parent2: CoreChromosome) -> CoreChromosome:

    # Takes half of Parent 1's steps (e.g. core center) and joins them with half of Parent 2's (edges).
    # NOTE: It will always cut exactly in the middle, so it doesn't vary that much
    # cut_point = len(parent1.steps) // 2

    # Defines a random cut point, to include more variation
    cut_point = random.randint(1, len(parent1.steps) - 1)

    # Copies the steps to avoid unwanted memory references
    child_steps = [Step(s.width, s.thickness) for s in parent1.steps[:cut_point]]
    child_steps += [Step(s.width, s.thickness) for s in parent2.steps[cut_point:]]

    return CoreChromosome(steps=child_steps)


def mutate(chromosome: CoreChromosome):
    """
    Iterates through the steps and applies small changes to width or thickness.
    """
    for step in chromosome.steps:
        if random.random() < MUTATION_RATE:
            # Randomly chooses whether to mutate the width or the thickness
            if random.random() < 0.5:
                # Mutates the width by up to +/- 5%
                step.width *= random.uniform(0.95, 1.05)
                # Discretizes the width to be a multiple of 10
                step.width = round(step.width / 10) * 10
                if step.width < 10.0: # Ensures a minimum width of 10mm
                    step.width = 10.0
            else:
                # Adds or removes a step-lap package
                variation = random.choice([-1, 1]) * STEP_LAP_THICKNESS
                step.thickness += variation
                if step.thickness < STEP_LAP_THICKNESS:
                    step.thickness = STEP_LAP_THICKNESS # Avoids zero or negative thickness

def normalize_widths(chromosome: CoreChromosome):
    """
    Ensures that the step widths are strictly decreasing and multiples of WIDTH_INTERVAL.
    """
    # Ensures the first step's width is a multiple and not too small
    chromosome.steps[0].width = max(WIDTH_INTERVAL, round(chromosome.steps[0].width / WIDTH_INTERVAL) * WIDTH_INTERVAL)

    for i in range(1, len(chromosome.steps)):
        current_step = chromosome.steps[i]
        previous_step = chromosome.steps[i-1]

        # First, ensures it's a multiple of WIDTH_INTERVAL and not below the minimum
        current_step.width = max(WIDTH_INTERVAL, round(current_step.width / WIDTH_INTERVAL) * WIDTH_INTERVAL)

        # Enforces the strict decrease constraint
        # If the current width is not strictly smaller than the previous one, adjust it
        while current_step.width >= previous_step.width:
            current_step.width -= WIDTH_INTERVAL
            # Ensures it doesn't go below the minimum allowed width
            if current_step.width < WIDTH_INTERVAL:
                current_step.width = WIDTH_INTERVAL
                break # Can't decrease any further, so stop

MAIN FUNCTION

In [ ]:
def optimize_core_design():
    # 1. Initial Settings
    RADIUS = CORE_DIAMETER / 2.0

    # Lists to monitor progress
    best_fitness_history = []
    average_fitness_history = []

    # 2. Initialize Population
    population = [
        generate_random_individual(RADIUS)
        for _ in range(POPULATION_SIZE)
    ]

    best_global_solution = None

    # Use a process pool to parallelize fitness evaluation
    # The number of processes can be adjusted to optimize CPU usage
    num_cores = multiprocessing.cpu_count()
    pool = multiprocessing.Pool(processes=num_cores)

    # 3. Evolutionary Loop
    for generation in range(GENERATIONS):

        # Step A: Evaluate the whole population in parallel
        # Build an argument list for starmap: [(individual1, RADIUS), (individual2, RADIUS), ...]
        fitness_args = [(individual, RADIUS) for individual in population]

        # starmap applies evaluate_fitness to each argument tuple and returns the fitness values
        fitness_scores = pool.starmap(evaluate_fitness, fitness_args)

        # Assign the scores back to the chromosomes and identify the best one
        for i, individual in enumerate(population):
            individual.fitness = fitness_scores[i]
            # Tracks the best of all time
            if best_global_solution is None or individual.fitness > best_global_solution.fitness:
                # It's important to make a deep copy of the best one
                best_global_solution = CoreChromosome(
                    steps=[Step(s.width, s.thickness) for s in individual.steps],
                    fitness=individual.fitness
                )
                normalize_widths(best_global_solution) # Normalizes the best solution found

        # Record data for the monitoring chart
        generation_best = max(fitness_scores)
        generation_average = sum(fitness_scores) / len(fitness_scores)
        best_fitness_history.append(generation_best)
        average_fitness_history.append(generation_average)

        new_population = []

        # Elitism: Pass the best individual directly to the next generation
        new_population.append(best_global_solution)

        # Step B: Create the new generation
        while len(new_population) < POPULATION_SIZE:
            # Selects two competent parents
            parent1 = tournament_selection(population)
            parent2 = tournament_selection(population)

            # Reproduces and creates a child
            child = single_point_crossover(parent1, parent2)

            # Undergoes mutation
            mutate(child)

            normalize_widths(child) # Normalizes the child's widths after crossover and mutation

            # Adds to the new generation
            new_population.append(child)

        # Updates the population for the next cycle
        population = new_population

        # (Optional) Progress log every 50 generations
        if generation % 50 == 0:
            print(f"Generation {generation} | Best Fitness (Useful Area): {best_global_solution.fitness:.2f} mm²")

    # Close the process pool
    pool.close()
    pool.join()

    # 4. Generate the Evolution Chart
    plt.figure(figsize=(10, 5))
    plt.plot(best_fitness_history, label='Best Fitness', color='green')
    plt.plot(average_fitness_history, label='Average Fitness', color='blue', linestyle='--')
    plt.title("Genetic Algorithm Evolution")
    plt.xlabel("Generation")
    plt.ylabel("Fitness (Useful Area mm²)")
    plt.legend()
    plt.grid(True)
    plt.show()

    # 5. Return the Result
    print("\nRESULTS:")

    # Efficiency indicator calculations
    circle_area = math.pi * (RADIUS**2)
    # Total geometric area (sum of each step's area * 2 to cover both sides)
    total_geometric_area = sum(s.width * s.thickness for s in best_global_solution.steps) * 2
    total_magnetic_area = total_geometric_area * STACKING_FACTOR

    occupied_percentage = (total_geometric_area / circle_area) * 100
    magnetic_percentage = (total_magnetic_area / circle_area) * 100

    print(f"Core diameter: {CORE_DIAMETER} mm")
    print(f"Stacking factor: {STACKING_FACTOR * 100:.1f} %")
    print(f"\nCircle area: {circle_area:.2f} mm²")
    print(f"Core Section Area (Geometric): {total_geometric_area:.2f} mm²")
    print(f"Magnetic Section Area: {total_magnetic_area:.2f} mm²")
    print(f"\nOccupied area percentage: {occupied_percentage:.2f} %")
    print(f"Magnetic section percentage: {magnetic_percentage:.2f} %")

    return best_global_solution

FUNCTION TO PLOT THE RESULT

In [ ]:
def plot_core(best_solution: CoreChromosome, diameter: float):
    radius = diameter / 2.0
    fig, ax = plt.subplots(figsize=(8, 8))

    # Draws the core's circle
    circle = plt.Circle((0, 0), radius, color='lightgray', fill=False, linestyle='--', label='Core Boundary')
    ax.add_patch(circle)

    current_y = 0
    # Draws each step (mirrored across the 4 quadrants)
    for step in best_solution.steps:
        width = step.width
        thickness = step.thickness

        # Rectangle centered on X, growing upward in Y
        # Rect: (bottom_left_x, bottom_left_y), width, height
        top_rect = plt.Rectangle((-width/2, current_y), width, thickness,
                                 edgecolor='navy', facecolor='royalblue', alpha=0.7)
        bottom_rect = plt.Rectangle((-width/2, -current_y - thickness), width, thickness,
                                 edgecolor='navy', facecolor='royalblue', alpha=0.7)

        ax.add_patch(top_rect)
        ax.add_patch(bottom_rect)
        current_y += thickness

    ax.set_xlim(-radius * 1.1, radius * 1.1)
    ax.set_ylim(-radius * 1.1, radius * 1.1)
    ax.set_aspect('equal')
    plt.title(f"Optimized Core Profile (Ø {diameter}mm)")
    plt.xlabel("Width (mm)")
    plt.ylabel("Thickness (mm)")
    plt.grid(True, which='both', linestyle=':', alpha=0.5)
    plt.legend()
    plt.show()

RUNNING THE CODE

In [ ]:
import pandas as pd

# Script entry point
if __name__ == "__main__":
    num_cpus = multiprocessing.cpu_count()
    print(f"Available CPU count: {num_cpus}\n")

    best_core = optimize_core_design()

    # Prepare data for the table
    steps_data = []
    for i, step in enumerate(best_core.steps):
        steps_data.append({"Step": i + 1, "Width (mm)": step.width, "Thickness (mm)": step.thickness})

    # Create the DataFrame and display it
    steps_df = pd.DataFrame(steps_data)
    print("\nBest Core Specifications:")
    display(steps_df)

    # Visualize the optimized core
    plot_core(best_core, CORE_DIAMETER)